# JEPA Pretraining — sweep over pretraining epochs

Tests whether the JEPA objective helps symbolic regression more as a
**representation-pretraining stage** than as an auxiliary loss.

```text
Stage 1  JEPA-only pretraining   (pretrain_epochs)
   |     encoder learns to predict a fixed symbolic target
   v
Stage 2  CE-only fine-tuning     (FINETUNE_EPOCHS)
```

Swept variable: `pretrain_epochs ∈ {0, 5, 10, 15}`. **`0` is the baseline** —
Stage 1 is skipped and the run is plain CE training.

There is **no lambda** here: Stage 1 optimises the bare JEPA loss, so lambda
would only rescale a loss that is the sole objective. This notebook writes to
its own checkpoint root and uses its own run-tag scheme, so it can neither
read nor overwrite results from `jepa_sweep.ipynb`.

In [ ]:
# Environment setup — works on both Colab and local
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone repo if not already present
    REPO_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa'
    if not os.path.exists(REPO_DIR):
        %cd /content/drive/MyDrive/Symba
        !git clone https://github.com/zzpDavid2/symbolic-jepa.git {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    !pip install -q sympy scipy

    # Checkpoint dir on Drive for persistence
    CKPT_DIR = '/content/drive/MyDrive/Symba/symbolic-jepa/checkpoints/jepa_pretrain'
    LOG_DIR  = '/content/drive/MyDrive/Symba/symbolic-jepa/runs_pretrain'
else:
    CKPT_DIR = 'checkpoints/jepa_pretrain'
    LOG_DIR  = 'runs_pretrain'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')
print(f'Checkpoints: {CKPT_DIR}')

In [ ]:
if IN_COLAB:
    %cd /content/drive/MyDrive/Symba/symbolic-jepa
    !git pull

In [ ]:
import gc
import torch
import torch.nn.functional as F
import numpy as np
import random
import datetime
import json
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from symbolic_jepa import (
    PrefixTokenizer, Expression,
    TNet, SymbolicTransformer,
    JEPAPredictor, IdentityPredictor, jepa_loss,
    PointCloudDataset, build_synthetic_splits,
    load_synthetic_pkl,
    teacher_forced_accuracy, teacher_forced_counts,
    evaluate_predictions, cleanup_eval_pool,
    sym_spread, pred_spread, retrieval_top1, common_mode,
)
from symbolic_jepa.tokenizer import prefix_to_sympy

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Model / training hyperparameters (identical to jepa_sweep) ──
MAX_VARS    = 1               # univariate synthetic data
D_INPUT     = MAX_VARS + 1    # (x, y) = 2
N_POINTS    = 1000
MAX_SEQ     = 64
D_MODEL     = 512
N_HEADS     = 8
N_LAYERS    = 4
DROPOUT     = 0.2

LR          = 3e-4
BATCH       = 16
VAL_EVERY   = 1
USE_AMP     = True

# DataLoader workers: 0.
#
# Under fork (Colab/Linux), workers are forked one at a time, so with
# num_workers >= 2 the SECOND worker inherits a heap copy of its own loader's
# iterator -- which by then already holds worker 0's Process object, tagged
# with the PARENT's pid. If that inherited object is ever finalised inside the
# child, Process.is_alive() trips:
#     AssertionError: can only test a child process
# It is a single loader inheriting from itself; it needs no second loader and
# no persistent_workers to happen, which is why restructuring the loaders did
# not stop it. Measured cost of 0 workers here: ~4 s/epoch (~10% of an epoch).
#
# num_workers = 1 is also safe for this mechanism (that worker inherits an
# EMPTY _workers list, and the pin-memory thread is created after the fork),
# and keeps the prefetch overlap. Raise to 1 if you want that ~10% back.
NUM_WORKERS = 0

# ── Stage 2 (CE fine-tuning) length ──
# This is the ONLY downstream epoch count. There is no separate EPOCHS here:
# a stray EPOCHS could silently override it, so it simply does not exist.
FINETUNE_EPOCHS = 30

# ── The swept variable: length of Stage 1 ──
# 0 = baseline (Stage 1 skipped -> plain CE training, the control).
PRETRAIN_EPOCHS_VALUES = [0, 5, 10, 15]
SEEDS                  = [42, 123, 7]

# JEPA objective used in Stage 1 (unchanged from jepa_sweep)
JEPA_LOSS      = 'cosine'    # 'cosine' (ordinary) | 'centered'
JEPA_PREDICTOR = 'identity'  # 'identity' | 'mlp'

# Stop-gradient on the symbolic target during Stage 1.
# Keep True: with False the JEPA-only objective collapses (see next cell).
PRETRAIN_STOPGRAD = True

# Own tag + own checkpoint root => cannot collide with the joint sweep.
VERSION_TAG = 'pretrain_v1'

# Bump whenever DECODING or SCORING changes, independently of training.
# Trained checkpoints stay valid; only metrics.json is invalidated, so the
# existing runs are re-evaluated rather than retrained.
#   v2_strict_eos: greedy freezes a row after its <eos>, decode() stops at the
#                  first <eos>, and prefix parsing rejects trailing tokens.
EVAL_VERSION = 'v2_strict_eos'

# Synthetic data (pre-generated by SYMBA_Reg_Data_Gen notebook)
SYNTH_PKL   = 'data/synthetic.pkl'
MAX_SYNTH   = 10_000
SYNTH_SEED  = 42

### Clear stale evaluation caches


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# COMMENTED OUT BY DEFAULT — uncomment the block below to run it.
#
# Removes evaluation caches (metrics.json) left by an OLDER evaluator, i.e.
# any whose eval_version != EVAL_VERSION. Checkpoints are never touched, so
# nothing is retrained; affected runs are simply re-scored.
#
# You do not normally need this: eval_one already re-evaluates on an
# eval_version mismatch. Use it only to force a clean slate.
# ══════════════════════════════════════════════════════════════════════
# _root = f'{CKPT_DIR}/{VERSION_TAG}'
# _stale, _kept = [], 0

# if not os.path.isdir(_root):
#     print(f'No run directory yet: {_root}')
# else:
#     for _run in sorted(os.listdir(_root)):
#         _mp = f'{_root}/{_run}/metrics.json'
#         if not os.path.exists(_mp):
#             continue
#         try:
#             with open(_mp) as _f:
#                 _ver = json.load(_f).get('eval_version')
#         except Exception as _e:
#             _ver = f'<unreadable: {type(_e).__name__}>'
#         if _ver == EVAL_VERSION:
#             _kept += 1
#         else:
#             _stale.append((_mp, _ver))

#     for _mp, _ver in _stale:
#         print(f'Deleting (eval_version={_ver!r}): {_mp}')
#         os.remove(_mp)

#     _fr = f'{_root}/failed_runs.json'
#     if _stale and os.path.exists(_fr):
#         print(f'Deleting: {_fr}')
#         os.remove(_fr)

#     print(f'\nRemoved {len(_stale)} stale cache(s); '
#           f'kept {_kept} at eval_version={EVAL_VERSION!r}.')
#     print('Checkpoints untouched — affected runs are re-scored, not retrained.')

### Why `PRETRAIN_STOPGRAD` must stay `True`

Stage 1 optimises the JEPA loss alone. In the joint sweep, `JEPA_STOPGRAD` was
`False`, so gradients reached **both** sides (`encoder`, `tok_embed`,
`pos_embed`, `transformer`, `norm` — and `head.weight` is tied to
`tok_embed.weight`). CE anchored both sides there. Remove CE and the pair is
free to converge on one constant vector: `cos = 1`, loss = 0, nothing learned.

Measured over 6 JEPA-only epochs (d_model=128, 458 equations):

| | `stopgrad=False` | `stopgrad=True` |
|---|---|---|
| JEPA loss | 1.028 → **0.0037** | 1.028 → 0.314 |
| std(z_sym) | 0.716 → **0.078** | 0.716 (frozen) |
| off-diag cos(z_sym) | 0.473 → **0.994** | 0.473 (frozen) |
| std(z_num) | 0.011 → 0.298 | 0.011 → **0.370** |
| retrieval@1 | ~chance | above chance |

`stopgrad=False` shows the full collapse signature — every equation lands on
nearly the same point and the loss is "solved" without learning anything.

**What this means for reading results:** with the stop-gradient, Stage 1 trains
**only the T-Net encoder** against a fixed, randomly-initialised symbolic
target. The decoder is not pretrained. So this measures *"does JEPA-pretraining
the point-cloud encoder help?"*, not *"does pretraining the whole model help?"*

`torch.no_grad()` alone is **not** sufficient — it blocks gradients but leaves
dropout active, which would make the "fixed" target re-sample every step. Stage 1
therefore puts the model in `eval()` while computing the target.

## Load synthetic data

In [ ]:
tokenizer = PrefixTokenizer(max_vars=MAX_VARS)
print(f'Vocab size: {len(tokenizer)}')

print(f'Loading synthetic expressions from {SYNTH_PKL}...')
synth_exprs = load_synthetic_pkl(
    SYNTH_PKL, max_seq_len=MAX_SEQ,
    tokenizer=tokenizer, max_expressions=MAX_SYNTH,
)
print(f'Loaded {len(synth_exprs)} expressions')

# Inspect a few
for expr in synth_exprs[:5]:
    print(f'  {expr.prefix}')

In [ ]:
# Split synthetic data
synth_train, synth_val, synth_test = build_synthetic_splits(
    synth_exprs, tokenizer,
    n_points=N_POINTS, max_seq_len=MAX_SEQ, max_vars=MAX_VARS,
    seed=SYNTH_SEED,
    # val/test clouds are deterministic; caching lets their loaders run with
    # num_workers=0 at no cost (see NUM_WORKERS note in the config cell).
    cache_eval=True,
)

## Training / evaluation

In [ ]:
from torch.utils.tensorboard import SummaryWriter

In [ ]:
import time

def seed_everything(seed):
    """Full re-seed for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed + worker_id)
    random.seed(worker_seed + worker_id)

def _recover_best_val_acc(ck_path):
    """Recover best-epoch val_acc from old checkpoints that lack best_val_acc.

    Finds the epoch with min val loss in history and returns its val_acc.
    """
    ck = torch.load(ck_path, map_location='cpu', weights_only=False)
    history = ck.get('history', {})
    val_losses = history.get('val', [])
    val_accs = history.get('val_acc', [])
    if not val_losses or not val_accs:
        return 0.0
    n = min(len(val_losses), len(val_accs))
    best_idx = int(np.argmin(val_losses[:n]))
    return val_accs[best_idx]

def make_run_tag(pretrain_epochs, seed):
    """THE single source of truth for run naming.

    Every checkpoint, metrics.json, TensorBoard dir, plot and diagnostic must
    derive its path from this -- never from a hand-built f-string. The scheme
    deliberately shares no prefix with the joint sweep's 'lam{lam}_seed{seed}',
    so a pretrain run can never read or overwrite an old joint result.
    """
    return f'pre{pretrain_epochs}_seed{seed}'


class ConfigMismatch(RuntimeError):
    """Raised when a cached checkpoint/metrics file was made by another config."""


def make_run_config(pretrain_epochs):
    """Every setting that materially changes what a run means.

    Stored in each checkpoint and metrics.json, and compared before anything
    cached is reused. The run tag encodes only (pretrain_epochs, seed), so
    without this a changed JEPA_LOSS / LR / architecture would silently
    resurrect results produced under the old settings.
    """
    return {
        'pretrain_epochs': pretrain_epochs,
        'finetune_epochs': FINETUNE_EPOCHS,
        'pretrain_stopgrad': PRETRAIN_STOPGRAD,
        'jepa_loss': JEPA_LOSS,
        'jepa_predictor': JEPA_PREDICTOR,
        'lr': LR,
        'batch': BATCH,
        'dropout': DROPOUT,
        'd_model': D_MODEL,
        'n_heads': N_HEADS,
        'n_layers': N_LAYERS,
        'max_seq': MAX_SEQ,
        'n_points': N_POINTS,
        'max_vars': MAX_VARS,
        'synth_seed': SYNTH_SEED,
        'max_synth': MAX_SYNTH,
    }


def validate_run_config(saved, current, path):
    """Raise ConfigMismatch unless `saved` matches `current` exactly.

    A missing/None `saved` means a legacy file with no provenance: also
    refused, since we cannot tell what produced it.
    """
    if not saved:
        raise ConfigMismatch(
            f'Legacy checkpoint has no run configuration metadata.\n'
            f'Refusing automatic reuse.\n  {path}\n'
            f'Delete it, or bump VERSION_TAG to start a clean run directory.')
    diff = {k: (saved.get(k, '<missing>'), v)
            for k, v in current.items() if saved.get(k, '<missing>') != v}
    extra = {k: saved[k] for k in saved if k not in current}
    if diff or extra:
        lines = [f'  {k}: stored={sv!r}  current={cv!r}'
                 for k, (sv, cv) in sorted(diff.items())]
        lines += [f'  {k}: stored={v!r}  current=<absent>'
                  for k, v in sorted(extra.items())]
        raise ConfigMismatch(
            'CONFIG MISMATCH\n'
            'Refusing to reuse cached checkpoint/metrics.\n'
            f'  {path}\n' + '\n'.join(lines) +
            '\nBump VERSION_TAG (or delete the run dir) to start fresh.')


def stage2_epoch_seed(seed, epoch):
    """Stage 2 RNG seed for one fine-tuning epoch.

    Depends only on (seed, epoch) -- never on how much RNG Stage 1 consumed,
    nor on which epochs ran in this process. So epoch N is bit-identical
    whether the run was uninterrupted or resumed straight into epoch N, and
    every pretrain_epochs condition at a given seed shares one trajectory.
    """
    return seed + 100_000 + epoch


def make_finetune_loader(seed, epoch, synth_train):
    """Deterministically build the Stage 2 loader for one epoch.

    Also reseeds the global RNG so dropout follows the same per-epoch stream.
    This is the only loader in the run that forks workers, which keeps
    DataLoader teardown well-defined.
    """
    es = stage2_epoch_seed(seed, epoch)
    seed_everything(es)
    g = torch.Generator()
    g.manual_seed(es)
    return DataLoader(synth_train, batch_size=BATCH, shuffle=True,
                      num_workers=NUM_WORKERS, persistent_workers=False,
                      pin_memory=True, worker_init_fn=seed_worker,
                      generator=g)


def train_one(pretrain_epochs, seed, synth_train, synth_val, tokenizer):
    """Train one (lambda, seed) run. No eval — just training + checkpoints."""
    seed_everything(seed)

    run_tag = make_run_tag(pretrain_epochs, seed)
    run_dir = f'{CKPT_DIR}/{VERSION_TAG}/{run_tag}'
    CKPT_PATH = f'{run_dir}/latest.pt'
    BEST_PATH = f'{run_dir}/best.pt'
    metrics_path = f'{run_dir}/metrics.json'

    # ── Primary completion marker: metrics.json ─────────────────────────
    # Only eval_one writes it, and only after Phase 1 finished this run, so
    # its presence means training completed. Preferred over latest.pt's epoch
    # because it is a few KB: an interrupted Drive sync can leave the ~215 MB
    # latest.pt at an older revision while metrics.json (and best.pt) survive.
    # Its run_config is still validated, so a real config change is not
    # silently skipped. eval_version is deliberately NOT checked here -- an
    # evaluator change does not invalidate training, and eval_one re-scores
    # on its own when the version differs.
    if os.path.exists(metrics_path):
        try:
            with open(metrics_path) as _f:
                _m = json.load(_f)
        except Exception as _e:
            print(f'\n{run_tag}: metrics.json unreadable ({type(_e).__name__}); '
                  f'falling back to checkpoint state')
            _m = None
        if _m is not None:
            validate_run_config(_m.get('run_config'),
                                make_run_config(pretrain_epochs), metrics_path)
            print(f'\n{run_tag}: already trained and evaluated '
                  f'(metrics.json present), SKIPPING')
            return

    # Skip if training already completed (latest.pt exists with final epoch)
    if os.path.exists(CKPT_PATH):
        ck = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
        validate_run_config(ck.get('run_config'),
                            make_run_config(pretrain_epochs), CKPT_PATH)

        # ── Guard: latest.pt must not be OLDER than best.pt ──────────────
        # best.pt is written before latest.pt within the same epoch, so a
        # single run always satisfies latest.epoch >= best.epoch. If that is
        # violated, the two files came from different sessions -- typically
        # latest.pt was lost/reverted (it is ~2.5x larger than best.pt, so an
        # interrupted Drive sync drops it first) while best.pt survived from a
        # completed run.
        #
        # Resuming here would restart from latest.pt's early-epoch WEIGHTS and,
        # because best_val is also reloaded from that stale file, the first
        # improvement would overwrite the good best.pt with a far worse one.
        # Refuse instead of silently destroying it.
        if os.path.exists(BEST_PATH):
            _b = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
            _be, _le = _b.get('epoch'), ck.get('epoch')
            if _be is not None and _le is not None and _be > _le:
                raise ConfigMismatch(
                    f'STALE latest.pt — REFUSING TO TRAIN\n'
                    f'  {run_dir}\n'
                    f'  best.pt   epoch={_be}  val={_b.get("val")}\n'
                    f'  latest.pt epoch={_le}  best_val={ck.get("best_val")}\n'
                    f'best.pt is NEWER than latest.pt, so they are from '
                    f'different sessions. Resuming would overwrite the good '
                    f'best.pt with a worse checkpoint.\n'
                    f'If metrics.json for this run is already correct, leave '
                    f'it alone. To retrain from scratch, move best.pt and '
                    f'latest.pt aside first.')
            del _b

        if ck['epoch'] >= FINETUNE_EPOCHS:
            print(f'\n{run_tag}: training complete (epoch {ck["epoch"]}), SKIPPING')
            return
    os.makedirs(run_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'{run_tag} | pretrain={pretrain_epochs}ep | finetune={FINETUNE_EPOCHS}ep '
          f'| jepa_loss={JEPA_LOSS} | stopgrad={PRETRAIN_STOPGRAD} | tag={VERSION_TAG}')
    print(f'{"="*60}')

    encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)
    model = SymbolicTransformer(
        encoder=encoder, vocab_size=len(tokenizer),
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
        d_ff=4 * D_MODEL, max_seq_len=MAX_SEQ,
        dropout=DROPOUT, pad_id=tokenizer.pad_id,
    ).to(DEVICE)

    # Always built: 'identity' is parameter-free, and Stage 1 / the val
    # diagnostics both need it.
    predictor = (JEPAPredictor(D_MODEL).to(DEVICE) if JEPA_PREDICTOR == 'mlp'
                 else IdentityPredictor().to(DEVICE))

    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(synth_train, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS,
                              persistent_workers=False, pin_memory=True,
                              worker_init_fn=seed_worker, generator=g)
    # num_workers=0: clouds are cached, and the Stage 2 loader is rebuilt
    # every epoch -- a persistent val iterator would be inherited by each new
    # fork and blow up in its __del__.
    val_loader = DataLoader(synth_val, batch_size=BATCH, shuffle=False,
                            num_workers=0)

    params = list(model.parameters())
    if predictor is not None:
        params += list(predictor.parameters())
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=FINETUNE_EPOCHS)

    start_epoch = 1
    best_val = float('inf')
    best_val_acc = 0.0
    history = {'train': [], 'val': []}

    if os.path.exists(CKPT_PATH):
        ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
        # Never resume onto a checkpoint built under different settings.
        validate_run_config(ck.get('run_config'),
                            make_run_config(pretrain_epochs), CKPT_PATH)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        start_epoch = ck['epoch'] + 1
        best_val = ck['best_val']
        best_val_acc = ck.get('best_val_acc', 0.0)
        history = ck['history']
        if predictor is not None and 'predictor' in ck:
            predictor.load_state_dict(ck['predictor'])
        print(f'  Resuming from epoch {start_epoch} (best val: {best_val:.4f})')

    if USE_AMP and DEVICE == 'cuda':
        amp_ctx = lambda: torch.autocast('cuda', dtype=torch.bfloat16)
    elif USE_AMP and DEVICE == 'mps':
        amp_ctx = lambda: torch.autocast('mps', dtype=torch.float16)
    else:
        amp_ctx = lambda: torch.amp.autocast('cpu', enabled=False)

    tb_dir = f'{LOG_DIR}/{VERSION_TAG}/{run_tag}'
    writer = SummaryWriter(log_dir=tb_dir)

    # ══════════════════════════════════════════════════════════════
    # Stage 1: JEPA-only pretraining (skipped when pretrain_epochs == 0)
    # ══════════════════════════════════════════════════════════════
    # Gradient flow here is JEPA -> encoder only (with PRETRAIN_STOPGRAD=True).
    # There is deliberately NO decoder forward pass and NO CE term: the loss
    # is the bare JEPA objective. There is no lambda in this experiment --
    # the swept variable is pretrain_epochs.
    #
    # pretrain_epochs == 0 is the BASELINE: Stage 1 is skipped entirely and
    # the run is plain CE training, which is the control for this sweep.
    # Also skipped when resuming (start_epoch > 1): Stage 1 already ran and
    # its weights are inside the checkpoint we just loaded.
    if pretrain_epochs > 0 and start_epoch == 1:
        print(f'\n=== Stage 1: JEPA pretraining '
              f'({pretrain_epochs} epochs, loss = JEPA only) ===')
        print(f'  stopgrad={PRETRAIN_STOPGRAD}  '
              f'trains={"encoder only" if PRETRAIN_STOPGRAD else "encoder + decoder"}')
        # Stage 1 gets its OWN optimizer; a fresh one is built for Stage 2 so
        # no Adam moments / LR schedule carry across the objective switch.
        pre_opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.1)

        for pe in range(1, pretrain_epochs + 1):
            model.train()
            if predictor is not None:
                predictor.train()
            pre_sum = pre_n = 0
            std_num_sum = std_sym_sum = 0.0
            pbar_p = tqdm(train_loader, leave=False,
                          desc=f'{run_tag} P{pe}/{pretrain_epochs}')
            for batch in pbar_p:
                points    = batch['points'].to(DEVICE, non_blocking=True)
                input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
                attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)

                pre_opt.zero_grad()
                with amp_ctx():
                    if PRETRAIN_STOPGRAD:
                        # FIX: torch.no_grad() blocks gradients but does NOT
                        # disable dropout. Without eval() the "fixed" target
                        # is re-sampled every step, so the online branch chases
                        # a moving, noisy target. eval() first, restore after.
                        model.eval()
                        with torch.no_grad():
                            z_sym = model.encode_expression(
                                input_ids, attn_mask=attn_mask)
                        model.train()
                    else:
                        # Collapse-demo path only (see the note above): target
                        # is trainable, so it cannot be put in eval mode.
                        z_sym = model.encode_expression(
                            input_ids, attn_mask=attn_mask)
                    # Online branch, in train mode. Encoder only -- there is no
                    # decoder forward here, so no CE term can leak in.
                    z_num = model.encoder(points)
                    loss_pre = jepa_loss(predictor(z_num), z_sym, mode=JEPA_LOSS)

                loss_pre.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if predictor is not None:
                    torch.nn.utils.clip_grad_norm_(predictor.parameters(), 1.0)
                pre_opt.step()

                pre_sum += loss_pre.item(); pre_n += 1
                # Averaged over the whole epoch, not just the last batch:
                # a per-batch reading jitters with which equations land in
                # the final batch, which would mask a real collapse trend.
                std_num_sum += z_num.detach().float().std(dim=0).mean().item()
                std_sym_sum += z_sym.detach().float().std(dim=0).mean().item()
                pbar_p.set_postfix({'jepa': f'{loss_pre.item():.4f}'})
            pbar_p.close(); del pbar_p

            # Collapse diagnostics: mean per-dim std of the embeddings.
            # Trending to 0 means the representation is degenerating to a
            # constant. With PRETRAIN_STOPGRAD=True, std(z_sym) should stay
            # flat -- the symbolic side is frozen.
            pre_avg = pre_sum / max(pre_n, 1)
            std_num = std_num_sum / max(pre_n, 1)
            std_sym = std_sym_sum / max(pre_n, 1)
            history.setdefault('pretrain_jepa', []).append(pre_avg)
            history.setdefault('pretrain_std_num', []).append(std_num)
            history.setdefault('pretrain_std_sym', []).append(std_sym)
            writer.add_scalar('pretrain/jepa_loss', pre_avg, pe)
            writer.add_scalar('pretrain/std_z_num', std_num, pe)
            writer.add_scalar('pretrain/std_z_sym', std_sym, pe)
            print(f'  P{pe}/{pretrain_epochs} | jepa={pre_avg:.4f} | '
                  f'std(z_num)={std_num:.4f} std(z_sym)={std_sym:.4f}')

        # ── Stage transition: keep weights, drop optimizer/scheduler state ──
        del pre_opt
        optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.1)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=FINETUNE_EPOCHS)

    # ══════════════════════════════════════════════════════════════
    # Stage 2: CE-only fine-tuning
    # ══════════════════════════════════════════════════════════════
    # Stage 1 consumes RNG (shuffling, dropout), so without an explicit reset a
    # pretrained run would enter Stage 2 on a different stochastic trajectory
    # than the pretrain_epochs=0 baseline at the same seed -- and part of any
    # downstream difference would be trajectory, not initialisation.
    #
    # The reset is PER EPOCH (see stage2_epoch_seed), not once up front, so a
    # run resumed at epoch N sees exactly what an uninterrupted run sees at
    # epoch N. Model weights are deliberately NOT reset.
    print(f'\n=== Stage 2: CE fine-tuning '
          f'({FINETUNE_EPOCHS} epochs, loss = CE only, '
          f'stage2 epoch seed = {seed} + 100000 + epoch) ===')

    for epoch in range(start_epoch, FINETUNE_EPOCHS + 1):
        # Rebuilt every epoch from (seed, epoch) alone -> resume-invariant.
        ft_loader = make_finetune_loader(seed, epoch, synth_train)
        model.train()
        if predictor is not None:
            predictor.train()
        train_loss_gen = 0
        pbar = tqdm(ft_loader, leave=False,
                    desc=f'{run_tag} E{epoch}/{FINETUNE_EPOCHS}')
        for batch in pbar:
            points    = batch['points'].to(DEVICE, non_blocking=True)
            input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
            attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            with amp_ctx():
                out = model(points, input_ids, attn_mask=attn_mask)
                loss_gen = out['loss']

                # Stage 2 is CE only. The JEPA term is not gated to zero, it
                # is simply absent -- there is no lambda in this experiment.
                loss = loss_gen

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if predictor is not None:
                torch.nn.utils.clip_grad_norm_(predictor.parameters(), 1.0)
            optimizer.step()

            train_loss_gen += loss_gen.item()
            pbar.set_postfix({'loss': f'{loss_gen.item():.4f}'})

            global_step = (epoch - 1) * len(ft_loader) + pbar.n
            writer.add_scalar('train/loss_step', loss_gen.item(), global_step)

        # tqdm.auto holds iter(ft_loader); drop it before the next epoch's
        # loader forks, so no worker inherits a live iterator.
        n_ft_batches = len(ft_loader)
        pbar.close(); del pbar

        scheduler.step()
        train_avg = train_loss_gen / n_ft_batches
        history['train'].append(train_avg)
        writer.add_scalar('train/loss_epoch', train_avg, epoch)
        writer.add_scalar('train/lr', scheduler.get_last_lr()[0], epoch)

        # ── Validation (token-weighted aggregation) ──
        if epoch % VAL_EVERY == 0 or epoch == FINETUNE_EPOCHS:
            model.eval()
            if predictor is not None:
                predictor.eval()
            val_loss_sum = 0.0     # sum of (batch_loss * n_valid_tokens)
            val_tokens_total = 0   # total valid (non-pad) target tokens
            acc_correct = 0.0      # total correct tokens across all batches
            acc_total = 0.0        # total valid tokens for accuracy
            val_align_sum = 0
            n_val_batches = 0
            diag_z_sym = None
            diag_z_pred = None
            with torch.no_grad(), amp_ctx():
                for batch in val_loader:
                    points    = batch['points'].to(DEVICE, non_blocking=True)
                    input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
                    attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)
                    out = model(points, input_ids, attn_mask=attn_mask)

                    # Token-weighted loss: weight by number of valid tokens
                    n_tok = out['n_tokens']
                    val_loss_sum += out['loss'].item() * n_tok
                    val_tokens_total += n_tok

                    # Token-weighted accuracy: accumulate counts
                    c, t = teacher_forced_counts(out['logits'], input_ids, tokenizer.pad_id)
                    acc_correct += c
                    acc_total += t

                    if predictor is not None:
                        z_sym_v = model.encode_expression(input_ids, attn_mask=attn_mask)
                        z_num_v = out['z_num']
                        z_pred_v = predictor(z_num_v)
                        val_align_sum += jepa_loss(z_pred_v, z_sym_v, mode=JEPA_LOSS).item()
                        if diag_z_sym is None:
                            diag_z_sym = z_sym_v
                            diag_z_pred = z_pred_v

                    n_val_batches += 1

            val_avg = val_loss_sum / max(val_tokens_total, 1)
            val_acc_avg = acc_correct / max(acc_total, 1)
            history['val'].append(val_avg)
            history.setdefault('val_acc', []).append(val_acc_avg)
            writer.add_scalar('val/loss', val_avg, epoch)
            writer.add_scalar('val/token_accuracy', val_acc_avg, epoch)

            extra = ''
            if predictor is not None:
                val_align_avg = val_align_sum / n_val_batches
                writer.add_scalar('val/jepa_raw', val_align_avg, epoch)
                history.setdefault('val_jepa_raw', []).append(val_align_avg)

                ss = sym_spread(diag_z_sym)
                ps = pred_spread(diag_z_pred)
                rt = retrieval_top1(diag_z_pred, diag_z_sym)
                cm = common_mode(diag_z_sym)

                writer.add_scalar('val/sym_spread_raw', ss['raw'], epoch)
                writer.add_scalar('val/sym_spread_cent', ss['centered'], epoch)
                writer.add_scalar('val/pred_spread', ps['raw'], epoch)
                writer.add_scalar('val/retrieval_top1', rt['centered'], epoch)
                writer.add_scalar('val/common_mode_ratio', cm['mean_norm_ratio'], epoch)

                extra = (f' | jepa={val_align_avg:.4f}'
                         f' | retr={rt["centered"]:.2f}')

            is_best = val_avg < best_val
            if is_best:
                best_val = val_avg
                best_val_acc = val_acc_avg
            flag = ' * best' if is_best else ''
            print(f'  E{epoch}/{FINETUNE_EPOCHS} | train={train_avg:.4f} | '
                  f'val={val_avg:.4f}{flag} | acc={val_acc_avg*100:.1f}%{extra}')

            if is_best:
                # Provenance mirrors latest.pt so best.pt can be validated on
                # its own. Note best_val_acc means "val_acc AT the epoch of
                # minimum val loss", not "maximum val_acc" -- is_best is
                # decided by val loss above.
                torch.save({
                    'model': model.state_dict(),
                    'epoch': epoch,
                    'val': val_avg,
                    'val_acc': val_acc_avg,
                    'seed': seed,
                    'run_config': make_run_config(pretrain_epochs),
                }, BEST_PATH)

        ckpt_dict = {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch': epoch,
            'best_val': best_val,
            'best_val_acc': best_val_acc,
            'history': history,
            'seed': seed,
            # All experiment settings live in ONE place; see make_run_config.
            'run_config': make_run_config(pretrain_epochs),
        }
        if predictor is not None and isinstance(predictor, JEPAPredictor):
            ckpt_dict['predictor'] = predictor.state_dict()
        torch.save(ckpt_dict, CKPT_PATH)

    writer.close()

    del train_loader, ft_loader, val_loader
    del model, encoder, predictor, optimizer, scheduler
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()


def eval_one(pretrain_epochs, seed, synth_test, tokenizer):
    """Load best checkpoint and run greedy eval on full test set. Returns metrics dict."""
    run_tag = make_run_tag(pretrain_epochs, seed)
    run_dir = f'{CKPT_DIR}/{VERSION_TAG}/{run_tag}'
    metrics_path = f'{run_dir}/metrics.json'
    BEST_PATH = f'{run_dir}/best.pt'
    CKPT_PATH = f'{run_dir}/latest.pt'

    # Reuse a cached evaluation only if it came from THIS training config AND
    # THIS evaluator. The two are versioned separately: a decoding/scoring
    # change leaves the trained weights valid but makes old metrics wrong.
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            cached = json.load(f)
        cached_eval = cached.get('eval_version')
        if cached_eval != EVAL_VERSION:
            print(f'  {run_tag}: evaluation cache version mismatch '
                  f'(cached={cached_eval!r}, current={EVAL_VERSION!r}). '
                  f'Re-evaluating checkpoint.')
        else:
            validate_run_config(cached.get('run_config'),
                                make_run_config(pretrain_epochs), metrics_path)
            return cached

    # Load best model
    encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)
    model = SymbolicTransformer(
        encoder=encoder, vocab_size=len(tokenizer),
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
        d_ff=4 * D_MODEL, max_seq_len=MAX_SEQ,
        dropout=0.0, pad_id=tokenizer.pad_id,
    ).to(DEVICE)

    # latest.pt is AUTHORITATIVE for validation metadata (it is the file
    # train_one validates and the only one carrying history); best.pt supplies
    # only the best model weights. Previously eval_one read val/val_acc from an
    # unvalidated best.pt, so a stale best.pt could silently populate the
    # summary table with numbers unrelated to the current run.
    latest_ck = None
    if os.path.exists(CKPT_PATH):
        latest_ck = torch.load(CKPT_PATH, map_location='cpu',
                               weights_only=False)
        validate_run_config(latest_ck.get('run_config'),
                            make_run_config(pretrain_epochs), CKPT_PATH)

    ck_path = BEST_PATH if os.path.exists(BEST_PATH) else CKPT_PATH
    ck = torch.load(ck_path, map_location=DEVICE, weights_only=False)

    # Validate best.pt too when it carries provenance. Older files predate
    # this and are allowed through, but must still agree numerically below.
    if ck_path == BEST_PATH:
        if ck.get('run_config') is not None:
            validate_run_config(ck['run_config'],
                                make_run_config(pretrain_epochs), BEST_PATH)
        if ck.get('seed') is not None and ck['seed'] != seed:
            raise ConfigMismatch(
                f'best.pt seed {ck["seed"]} != run seed {seed}\n  {BEST_PATH}')
        # Fail loudly rather than silently mixing two runs' numbers.
        if latest_ck is not None and ck.get('val') is not None:
            _lbv = latest_ck.get('best_val')
            if _lbv is not None and not np.isclose(ck['val'], _lbv,
                                                   rtol=1e-6, atol=1e-8):
                raise ConfigMismatch(
                    f'CHECKPOINT INCONSISTENCY\n'
                    f'  best.pt val      = {ck["val"]}\n'
                    f'  latest.best_val  = {_lbv}\n'
                    f'  {run_dir}\n'
                    f'best.pt does not belong to this run. Re-run training '
                    f'for this tag, or delete best.pt to fall back to '
                    f'latest.pt.')

    model.load_state_dict(ck['model'])
    model.eval()

    # Validation metadata: ONE source of truth = latest.pt, which train_one
    # validated. best.pt is only consulted if latest.pt is absent.
    if latest_ck is not None:
        best_val = latest_ck.get('best_val', float('inf'))
        best_val_acc = latest_ck.get('best_val_acc', None)
        if best_val_acc is None or best_val_acc == 0.0:
            best_val_acc = _recover_best_val_acc(CKPT_PATH)
    else:
        best_val = ck.get('val', float('inf'))
        best_val_acc = ck.get('val_acc', 0.0)

    # Greedy decode on FULL test set
    eval_loader = DataLoader(synth_test, batch_size=BATCH, shuffle=False)

    greedy_preds = []
    for batch in tqdm(eval_loader, desc=f'{run_tag} decode', leave=False):
        points = batch['points'].to(DEVICE)
        input_ids = batch['input_ids']
        preds = model.generate(points, tokenizer, max_new_tokens=MAX_SEQ)
        for j, pred_str in enumerate(preds):
            gt_str = tokenizer.decode(input_ids[j].tolist())
            greedy_preds.append((gt_str, pred_str))

    greedy_results = evaluate_predictions(greedy_preds, synth_test, tokenizer)

    metrics = {
        'seed': seed,
        'run_tag': run_tag,
        'pretrain_epochs': pretrain_epochs,
        'version_tag': VERSION_TAG,
        'run_config': make_run_config(pretrain_epochs),
        'eval_version': EVAL_VERSION,
        'best_val_loss': best_val,
        'best_val_acc': best_val_acc,
        'greedy_exact_match': greedy_results['exact_match'],
        'greedy_token_acc': greedy_results['token_accuracy'],
        'greedy_algebraic_equiv': greedy_results['algebraic_equiv'],
        'greedy_r2_above_0.9': greedy_results['r2_above_0.9'],
        'mean_r2': greedy_results['mean_r2'],
        'median_r2': greedy_results['median_r2'],
        'n_parseable': greedy_results['n_parseable'],
        'n_total': greedy_results['n_total'],
        'details': greedy_results['details'],
    }
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    del model, encoder, eval_loader
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    return metrics


# ══════════════════════════════════════════════════════════════
# Phase 1: Train all runs (no sympy, no stuck threads)
# ══════════════════════════════════════════════════════════════
print(f'Phase 1: Training {len(PRETRAIN_EPOCHS_VALUES)*len(SEEDS)} runs...')
for _pe in PRETRAIN_EPOCHS_VALUES:
    for _seed in SEEDS:
        train_one(_pe, _seed, synth_train, synth_val, tokenizer)

# ══════════════════════════════════════════════════════════════
# Phase 2: Evaluate all runs on full test set
# ══════════════════════════════════════════════════════════════
print(f'\n\n{"="*70}')
print(f'Phase 2: Evaluating all runs on full test set ({len(synth_test)} eqs)...')
print(f'{"="*70}')
_RUNS = [(p, s) for p in PRETRAIN_EPOCHS_VALUES for s in SEEDS]

# An evaluation crash (SymPy timeout, OOM, bug) is an INFRASTRUCTURE failure,
# not a model that scored 0. Writing zeros into the table would drag that
# condition's averages down and make it look like a worse model. Successes and
# failures are therefore kept apart, and averages use successes only.
successful_metrics = []
failed_runs = []

for i_run, (_pe, _seed) in enumerate(_RUNS):
    run_tag = make_run_tag(_pe, _seed)
    t0 = time.time()
    try:
        m = eval_one(_pe, _seed, synth_test, tokenizer)
        elapsed = time.time() - t0
        print(f'  [{i_run+1}/{len(_RUNS)}] {run_tag}: '
              f'exact={m["greedy_exact_match"]*100:.1f}% | '
              f'equiv={m["greedy_algebraic_equiv"]*100:.1f}% | '
              f'R²>.9={m["greedy_r2_above_0.9"]*100:.1f}% | '
              f'{elapsed:.0f}s')
        successful_metrics.append(m)
    except Exception as e:
        elapsed = time.time() - t0
        print(f'  [{i_run+1}/{len(_RUNS)}] {run_tag}: '
              f'FAILED after {elapsed:.0f}s — {type(e).__name__}: {e}')
        failed_runs.append({'pretrain_epochs': _pe, 'seed': _seed,
                            'run_tag': run_tag,
                            'error': f'{type(e).__name__}: {e}'})

    gc.collect()
    time.sleep(3)

# ── Per-run table (successful evaluations only) ──
print(f'\n{"="*70}')
print(f'Sweep complete — {VERSION_TAG} (n_test={len(synth_test)})')
print(f'{"="*70}')
print(f'\n{"pre_ep":>7} {"seed":>6} {"val_loss":>10} {"val_acc":>10} '
      f'{"exact":>8} {"equiv":>8} {"R²>.9":>8}')
print('-' * 65)
for m in successful_metrics:
    print(f'{m["pretrain_epochs"]:>7} {m["seed"]:>6} {m["best_val_loss"]:>10.4f} '
          f'{m.get("best_val_acc",0)*100:>9.1f}% '
          f'{m["greedy_exact_match"]*100:>7.1f}% '
          f'{m["greedy_algebraic_equiv"]*100:>7.1f}% '
          f'{m["greedy_r2_above_0.9"]*100:>7.1f}%')

# ── Averages per pretrain-epoch setting (pre_ep=0 is the baseline) ──
# n_failed is shown so a condition evaluated on fewer seeds is never mistaken
# for one that simply scored differently.
print(f'\n{"pre_ep":>7} {"avg_exact":>10} {"avg_equiv":>10} {"avg_R²>.9":>10} '
      f'{"avg_val_acc":>12} {"n_success":>10} {"n_failed":>9}')
print('-' * 74)
for _pe in PRETRAIN_EPOCHS_VALUES:
    runs = [m for m in successful_metrics if m['pretrain_epochs'] == _pe]
    n_fail = sum(1 for f in failed_runs if f['pretrain_epochs'] == _pe)
    if not runs:
        print(f'{_pe:>7} {"—":>10} {"—":>10} {"—":>10} {"—":>12} '
              f'{0:>10} {n_fail:>9}')
        continue
    print(f'{_pe:>7} '
          f'{np.mean([m["greedy_exact_match"] for m in runs])*100:>9.1f}% '
          f'{np.mean([m["greedy_algebraic_equiv"] for m in runs])*100:>9.1f}% '
          f'{np.mean([m["greedy_r2_above_0.9"] for m in runs])*100:>9.1f}% '
          f'{np.mean([m.get("best_val_acc", 0) for m in runs])*100:>11.1f}% '
          f'{len(runs):>10} {n_fail:>9}')

if failed_runs:
    print(f'\n{"="*70}')
    print(f'{len(failed_runs)} FAILED EVALUATION(S) — excluded from all averages')
    print(f'{"="*70}')
    for f in failed_runs:
        print(f'  {f["run_tag"]}: {f["error"]}')
    with open(f'{CKPT_DIR}/{VERSION_TAG}/failed_runs.json', 'w') as fh:
        json.dump(failed_runs, fh, indent=2)
    print(f'\n  saved -> {CKPT_DIR}/{VERSION_TAG}/failed_runs.json')
else:
    print('\nAll evaluations succeeded.')

# Kept for downstream cells that expect the old name.
all_metrics = successful_metrics


## Stage 1 diagnostics

In [ ]:
import matplotlib.pyplot as plt

def _load_ckpt(pretrain_epochs, seed, which='latest'):
    """Load a run checkpoint. Path ALWAYS via make_run_tag."""
    p = f'{CKPT_DIR}/{VERSION_TAG}/{make_run_tag(pretrain_epochs, seed)}/{which}.pt'
    if not os.path.exists(p):
        return None
    return torch.load(p, map_location='cpu', weights_only=False)

_runs = [(p, s, ck) for p in PRETRAIN_EPOCHS_VALUES for s in SEEDS
         if (ck := _load_ckpt(p, s)) is not None]

if not _runs:
    print('No runs found yet — run the sweep cell first.')
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for p, s, ck in _runs:
        h = ck['history']
        if not h.get('pretrain_jepa'):
            continue                      # baseline: no Stage 1
        pe = range(1, len(h['pretrain_jepa']) + 1)
        ax[0].plot(pe, h['pretrain_jepa'], marker='o', label=f'pre{p} s={s}')
        ax[1].plot(pe, h['pretrain_std_num'], marker='o', label=f'pre{p} s={s} z_num')
        ax[1].plot(pe, h['pretrain_std_sym'], marker='s', ls='--', alpha=.5,
                   label=f'pre{p} s={s} z_sym')
    ax[0].set_title('Stage 1: JEPA loss'); ax[0].set_xlabel('pretrain epoch')
    ax[1].set_title('Stage 1: embedding std  (→0 = collapse)')
    ax[1].set_xlabel('pretrain epoch')
    for a in ax:
        a.grid(alpha=.3); a.legend(fontsize=7)
    plt.tight_layout(); plt.show()

## Stage 2 learning curves

In [ ]:
# CE fine-tuning curves, grouped by pretrain length. JEPA loss is a different
# objective on a different scale and is plotted separately above.
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
_cmap = plt.get_cmap('viridis')
_col = {p: _cmap(i / max(len(PRETRAIN_EPOCHS_VALUES) - 1, 1))
        for i, p in enumerate(PRETRAIN_EPOCHS_VALUES)}

for p in PRETRAIN_EPOCHS_VALUES:
    for s in SEEDS:
        ck = _load_ckpt(p, s)
        if ck is None:
            continue
        h = ck['history']
        lbl = f'pre{p} s={s}' + (' (baseline)' if p == 0 else '')
        ep = range(1, len(h['train']) + 1)
        ax[0].plot(ep, h['train'], color=_col[p], alpha=.6, label=lbl)
        if h.get('val'):
            ax[1].plot(range(1, len(h['val']) + 1), h['val'],
                       color=_col[p], alpha=.6, label=lbl)
        if h.get('val_acc'):
            ax[2].plot(range(1, len(h['val_acc']) + 1),
                       [a * 100 for a in h['val_acc']],
                       color=_col[p], alpha=.6, label=lbl)

ax[0].set_title('Stage 2: train CE'); ax[1].set_title('Stage 2: val loss')
ax[2].set_title('Stage 2: val token acc %')
for a in ax:
    a.set_xlabel('finetune epoch'); a.grid(alpha=.3); a.legend(fontsize=6)
plt.tight_layout(); plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## Inspect predictions

In [ ]:
def to_infix(prefix_str):
    """Convert prefix string to infix via SymPy. Returns (infix_str, error_msg)."""
    try:
        expr, constants = prefix_to_sympy(prefix_str)
        return str(expr), None
    except Exception as e:
        return None, str(e)

def inspect_run(run_tag, n_show=20):
    """Load metrics.json for a run and display predictions."""
    metrics_path = f'{CKPT_DIR}/{VERSION_TAG}/{run_tag}/metrics.json'
    if not os.path.exists(metrics_path):
        print(f'{run_tag}: no metrics.json found')
        return None
    with open(metrics_path) as f:
        metrics = json.load(f)

    details = metrics.get('details', [])
    if not details:
        print(f'{run_tag}: no prediction details saved')
        return metrics

    print(f'\n{"="*80}')
    print(f'{run_tag} | exact={metrics["greedy_exact_match"]*100:.1f}% | equiv={metrics["greedy_algebraic_equiv"]*100:.1f}%')
    print(f'{"="*80}')

    for i, d in enumerate(details[:n_show]):
        gt_infix, _ = to_infix(d['gt'])
        pred_infix, pred_err = to_infix(d['pred'])

        r2_str = f'{d["r2"]:.4f}' if d['r2'] is not None else 'N/A'
        parseable = d.get('parseable', pred_err is None)
        equiv = d.get('equiv', False)
        if d['exact']:
            status = 'EXACT'
        elif equiv:
            status = 'EQUIV'
        elif parseable:
            status = 'PARSEABLE'
        else:
            status = 'UNPARSEABLE'

        print(f'\n── [{i}] {status} | R²={r2_str} ──')
        print(f'  GT:   {gt_infix}')
        if pred_err is None:
            print(f'  Pred: {pred_infix}')
        else:
            print(f'  Pred: PARSE FAILED: {pred_err}')
            print(f'        {d["pred"]}')

    total = len(details)
    n_unparseable = sum(1 for d in details if not d.get('parseable', True))
    n_exact = sum(1 for d in details if d['exact'])
    n_equiv = sum(1 for d in details if d.get('equiv', False))
    print(f'\n  Total: {total} | Exact: {n_exact} | Equiv: {n_equiv} | Unparseable: {n_unparseable}')

In [ ]:
# One run per pretrain setting, first seed. Path via make_run_tag only.
for _p in PRETRAIN_EPOCHS_VALUES:
    inspect_run(make_run_tag(_p, SEEDS[0]), n_show=10)

## R² diagnostics: baseline (`pre0`) vs a pretrained setting

Paired per-equation comparison — does pretraining shift the R² distribution?

In [ ]:
# ── Config ──
PRE_BASE = 0    # baseline: no pretraining
PRE_JEPA = 5    # pretrained setting to compare against

def _load_details(pretrain_epochs, seed):
    """Per-equation eval details. Path ALWAYS via make_run_tag."""
    p = f'{CKPT_DIR}/{VERSION_TAG}/{make_run_tag(pretrain_epochs, seed)}/metrics.json'
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return json.load(f).get('details', [])

def _load_r2_vec(pretrain_epochs, seed):
    d = _load_details(pretrain_epochs, seed)
    return None if d is None else [x.get('r2') for x in d]

def _classify(r2):
    """Bucket a held-out R2 value.

    NOTE 'high_r2' means numerically near-perfect on the sampled points --
    it is NOT the same as `greedy_algebraic_equiv`, which is a genuine
    symbolic/SymPy equivalence check. A prediction can have R2 >= 0.999 and
    still be a different function, so the two must not share a name.
    """
    if r2 is None or not np.isfinite(r2) or r2 < 0.9:
        return 'bad'
    return 'close' if r2 < 0.999 else 'high_r2'

bin_edges = ['failed/<0', '0-0.9', '0.9-0.99', '0.99-0.999', '>=0.999']

def _bin_r2(r2):
    if r2 is None or not np.isfinite(r2) or r2 < 0:
        return 'failed/<0'
    if r2 < 0.9:   return '0-0.9'
    if r2 < 0.99:  return '0.9-0.99'
    if r2 < 0.999: return '0.99-0.999'
    return '>=0.999'

# ── 1. R² histogram, aggregated over seeds ──
cb = {b: 0 for b in bin_edges}; cj = {b: 0 for b in bin_edges}
n_seeds_used = 0
for s in SEEDS:
    rb, rj = _load_r2_vec(PRE_BASE, s), _load_r2_vec(PRE_JEPA, s)
    if rb is None or rj is None:
        continue
    n_seeds_used += 1
    for r in rb: cb[_bin_r2(r)] += 1
    for r in rj: cj[_bin_r2(r)] += 1

if n_seeds_used == 0:
    print(f'No paired runs for pre{PRE_BASE} vs pre{PRE_JEPA} — run the sweep first.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(bin_edges)); w = 0.35
    ax.bar(x - w/2, [cb[b] for b in bin_edges], w, label=f'pre{PRE_BASE} (baseline)', alpha=.8)
    ax.bar(x + w/2, [cj[b] for b in bin_edges], w, label=f'pre{PRE_JEPA}', alpha=.8)
    ax.set_xticks(x); ax.set_xticklabels(bin_edges, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('# equations (summed over seeds)'); ax.legend(); ax.grid(axis='y', alpha=.3)
    ax.set_title(f'Held-out R²: pre{PRE_BASE} vs pre{PRE_JEPA}  ({n_seeds_used} seeds)')
    plt.tight_layout(); plt.show()

    # ── 2. Transition table ──
    classes = ['bad', 'close', 'high_r2']
    labels = {'bad': 'R²<0.9', 'close': '0.9≤R²<0.999', 'high_r2': 'R²≥0.999'}
    T = np.zeros((3, 3), dtype=int)
    for s in SEEDS:
        rb, rj = _load_r2_vec(PRE_BASE, s), _load_r2_vec(PRE_JEPA, s)
        if rb is None or rj is None:
            continue
        for a, b in zip(rb, rj):
            T[classes.index(_classify(a)), classes.index(_classify(b))] += 1

    n_total = T.sum()
    print(f'Transition table (rows=pre{PRE_BASE}, cols=pre{PRE_JEPA}, '
          f'{n_seeds_used} seeds, n={n_total})')
    print("Buckets are held-out R2 only -- 'high_r2' is NOT algebraic "
          "equivalence.\n")
    print(f'{"":>16}' + ''.join(f'{labels[c]:>16}' for c in classes) + f'{"total":>10}')
    for i, rc in enumerate(classes):
        print(f'{labels[rc]:>16}' + ''.join(f'{T[i,j]:>16}' for j in range(3))
              + f'{T[i,:].sum():>10}')
    print(f'{"total":>16}' + ''.join(f'{T[:,j].sum():>16}' for j in range(3))
          + f'{n_total:>10}')

    print('\nKey transitions:')
    for lbl, cnt in [('bad -> close', T[0,1]), ('bad -> high_r2', T[0,2]),
                     ('close -> bad', T[1,0]), ('close -> high_r2', T[1,2]),
                     ('high_r2 -> close', T[2,1]), ('high_r2 -> bad', T[2,0])]:
        print(f'  {lbl:>20}: {cnt:>5}  ({cnt/max(n_total,1)*100:.1f}%)')
    up = T[0,1] + T[0,2] + T[1,2]; dn = T[1,0] + T[2,0] + T[2,1]
    print(f'\n  improved: {up}   degraded: {dn}   net: {up-dn:+d}')

## Experiment summary

In [ ]:
def summarise(pretrain_epochs, seed):
    tag = make_run_tag(pretrain_epochs, seed)
    d = f'{CKPT_DIR}/{VERSION_TAG}/{tag}'
    ck = _load_ckpt(pretrain_epochs, seed)
    mp = f'{d}/metrics.json'
    m = json.load(open(mp)) if os.path.exists(mp) else None
    if ck is None and m is None:
        print(f'{tag}: nothing saved yet'); return

    h = (ck or {}).get('history', {})
    pj = h.get('pretrain_jepa') or []
    print(f'\n{"="*54}\n{tag}\n{"="*54}')
    print(f'  Seed                   : {seed}')
    print(f'  JEPA pretrain epochs   : {pretrain_epochs}'
          f'{"   (baseline: no Stage 1)" if pretrain_epochs == 0 else ""}')
    if pretrain_epochs:
        print(f'  Pretrain stopgrad      : {(m or ck).get("pretrain_stopgrad", "?")}')
        print(f'  Final JEPA pretrain    : {pj[-1]:.4f}' if pj else
              '  Final JEPA pretrain    : N/A')
        if h.get('pretrain_std_num'):
            print(f'  Final std(z_num)       : {h["pretrain_std_num"][-1]:.4f}'
                  f'    std(z_sym): {h["pretrain_std_sym"][-1]:.4f}')
    print(f'  Fine-tuning epochs     : {ck["epoch"] if ck else "?"} / {FINETUNE_EPOCHS}')
    print(f'  Stage 2 seed           : {(ck or {}).get("stage2_seed", "?")}')
    if ck:
        print(f'  Best val loss          : {ck.get("best_val", float("nan")):.4f}')
        print(f'  Best val token acc     : {ck.get("best_val_acc", 0)*100:.2f}%')
    if m:
        print(f'  Exact                  : {m["greedy_exact_match"]*100:.1f}%')
        print(f'  Equivalent             : {m["greedy_algebraic_equiv"]*100:.1f}%')
        print(f'  R² > 0.9               : {m["greedy_r2_above_0.9"]*100:.1f}%')
    else:
        print('  (test metrics: run Phase 2 first)')

for _p in PRETRAIN_EPOCHS_VALUES:
    for _s in SEEDS:
        summarise(_p, _s)

## Diagnostic: post-EOS garbage check

In [ ]:
# ── Diagnostic: did post-EOS garbage explain a run's low exact-match? ──
# Re-decodes a few test equations from an existing checkpoint and shows the
# raw generation next to the EOS-truncated prediction. Lightweight: loads one
# checkpoint and decodes N examples, no training.
SUSPICIOUS_SEEDS = [1618, 8675309]   # from the task; edit as needed


def diagnose_run(pretrain_epochs, seed, n_show=6):
    tag = make_run_tag(pretrain_epochs, seed)
    d = f'{CKPT_DIR}/{VERSION_TAG}/{tag}'
    ckpt = f'{d}/best.pt' if os.path.exists(f'{d}/best.pt') else f'{d}/latest.pt'
    if not os.path.exists(ckpt):
        print(f'{tag}: no checkpoint found — skipping')
        return

    encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)
    model = SymbolicTransformer(
        encoder=encoder, vocab_size=len(tokenizer), d_model=D_MODEL,
        n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=4 * D_MODEL,
        max_seq_len=MAX_SEQ, dropout=0.0, pad_id=tokenizer.pad_id).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE,
                                     weights_only=True)['model'])
    model.eval()

    from torch.utils.data import Subset
    loader = DataLoader(Subset(synth_test, range(n_show)), batch_size=n_show)
    batch = next(iter(loader))
    points = batch['points'].to(DEVICE)

    # Raw ids straight out of greedy, before any truncation.
    with torch.no_grad():
        ids = torch.full((points.shape[0], 1), tokenizer.sos_id,
                         dtype=torch.long, device=DEVICE)
        done = torch.zeros(points.shape[0], dtype=torch.bool, device=DEVICE)
        for _ in range(MAX_SEQ):
            nxt = model(points, ids)['logits'][:, -1, :].argmax(dim=-1)
            nxt = torch.where(done, torch.full_like(nxt, tokenizer.eos_id), nxt)
            done = done | nxt.eq(tokenizer.eos_id)
            ids = torch.cat([ids, nxt.unsqueeze(1)], dim=1)
            if done.all():
                break

    print(f'\n{"="*74}\n{tag}   ({os.path.basename(ckpt)})\n{"="*74}')
    n_post_eos = 0
    for j in range(points.shape[0]):
        raw_ids = ids[j].tolist()
        raw_toks = [tokenizer.id2token.get(int(t), '<unk>') for t in raw_ids]
        pred = tokenizer.decode(raw_ids)                  # stops at first <eos>
        gt = tokenizer.decode(batch['input_ids'][j].tolist())

        # What the OLD decoder would have produced: drop <eos>, keep the rest.
        legacy = ' '.join(t for t in raw_toks
                          if t not in ('<pad>', '<sos>', '<eos>'))
        had_garbage = legacy != pred
        n_post_eos += had_garbage

        try:
            equiv = equations_equivalent(pred, gt)
        except Exception:
            equiv = False
        try:
            prefix_to_sympy(pred)
            parses = True
        except Exception as e:
            parses = f'NO ({type(e).__name__})'

        print(f'\n[{j}] exact={pred.strip() == gt.strip()}  '
              f'equiv={equiv}  parses={parses}')
        print(f'  GT        : {gt}')
        print(f'  pred      : {pred}')
        if had_garbage:
            print(f'  OLD decode: {legacy}')
            print(f'  ^ post-EOS garbage the previous evaluator would have scored')
    print(f'\n  {n_post_eos}/{points.shape[0]} example(s) had post-EOS text '
          f'that the old decoder would have included.')

    del model, encoder
    gc.collect()


for _s in SUSPICIOUS_SEEDS:
    for _pe in PRETRAIN_EPOCHS_VALUES:
        diagnose_run(_pe, _s)